In [ ]:
import numpy as np
from scipy.interpolate import make_lsq_spline, make_smoothing_spline,  splprep, splev 
import os
from PIL import Image
import yaml
import pandas as pd
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from utils import nearest_spline_sample

In [ ]:
MAP_NAME = 'berlin'

path = f'../data/maps/{MAP_NAME}/{MAP_NAME}'
if os.path.exists(path+".png"):
    map_img_path = path+".png"
elif os.path.exists(path+".pgm"):
    map_img_path = path+".pgm"
else:
    raise Exception("Map not found!")

map_yaml_path = path+".yaml"
map_img = np.array(Image.open(map_img_path).transpose(Image.FLIP_TOP_BOTTOM))
map_img = map_img.astype(np.float64)

# load map yaml
with open(map_yaml_path, 'r') as yaml_stream:
    try:
        map_metadata = yaml.safe_load(yaml_stream)
        map_resolution = map_metadata['resolution']
        origin = map_metadata['origin']
    except yaml.YAMLError as ex:
        print(ex)

# calculate map parameters
orig_x = origin[0]
orig_y = origin[1]

In [ ]:
df = pd.read_csv(path+'_raceline.csv', header=0, comment='#', sep=';')
xref = df.iloc[:, 1].to_numpy(dtype=float)
yref = df.iloc[:, 2].to_numpy(dtype=float)
# Scale
xref -= orig_x
yref -= orig_y
xref /= map_resolution
yref /= map_resolution
# # Cut off last points
# xref = xref[:-1]
# yref = yref[:-1]

In [ ]:
tck, u = splprep((xref, yref), k=3, per=1, s=20.0)

fig, ax = plt.subplots()
ax.imshow(map_img, cmap='gray', origin='lower')
ax.plot(xref, yref)

line, = ax.plot([], [], 'go', ms=5)   # visible point

u0, u1 = u[0], u[-1]+.1

def update(theta):
    theta = u0 + ((theta - u0))   # wrap around closed spline
    x, y = splev(theta, tck)
    line.set_data([x], [y])
    return (line,)

thetas = np.linspace(u0, u1, 500, endpoint=False)
ani = FuncAnimation(fig, update, frames=thetas, interval=20, blit=True)

plt.show()

In [ ]:
ub = np.sum(np.hypot(np.diff(xref), np.diff(yref)))
u, tck = splprep((xref, yref), ub=ub)

clicked_points = []
fig, ax = plt.subplots()
ax.imshow(map_img, cmap='gray', origin='lower')
scatter, = ax.plot([], [], 'ro', ms=1.0)
points = []

def redraw():
    if points:
        xy = np.array(points)
        scatter.set_data(xy[:, 0], xy[:, 1])
    else:
        scatter.set_data([], [])

def onclick(event):
    if event.inaxes != ax or event.xdata is None or event.ydata is None:
        return
    x, y = event.xdata, event.ydata
    points.clear()
    points.append([x, y])
    # Find nearest
    points.append(nearest_spline_sample)
    redraw()

        

cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.show()